In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()

xTrain,xTest,yTrain,yTest = train_test_split(housing.data, housing.target)
xTrain,xVal,yTrain,yVal = train_test_split(xTrain,yTrain)

scaler = StandardScaler()
xTrain = torch.tensor(scaler.fit_transform(xTrain)).to(torch.float32)
xTest = torch.tensor(scaler.transform(xTest)).to(torch.float32)
xVal = torch.tensor(scaler.transform(xVal)).to(torch.float32)
yTrain = torch.tensor(yTrain).to(torch.float32)
yTest = torch.tensor(yTest).to(torch.float32)

In [3]:
import torch
from torch import nn
import torch.nn.functional as F

class Network(nn.Module): # inherits nn.Module
    def __init__(self):
        super().__init__() # WHYY??
        inputs = xVal.shape[1]

        self.hl1 = nn.Linear(inputs,20) # This type of linking similar to tf functional linking
        self.hl2 = nn.Linear(20,10)
        self.output = nn.Linear(10,1)

    def forwardProp(self,X):
        X = F.relu(self.hl1(X))
        X = F.relu(self.hl2(X))
        X = F.relu(self.output(X))

        return X

In [4]:
torch.manual_seed(42) # Seeds RNG

model = Network()
print(model)

Network(
  (hl1): Linear(in_features=8, out_features=20, bias=True)
  (hl2): Linear(in_features=20, out_features=10, bias=True)
  (output): Linear(in_features=10, out_features=1, bias=True)
)


In [5]:
lossFn = nn.MSELoss()
optimizer = torch.optim.NAdam(model.parameters())

In [6]:
n_epoch = 50
losses = []

for i in range(n_epoch):
    optimizer.zero_grad() # resets gradients to 0

    yPreds = model.forwardProp(xTrain)
    
    loss = lossFn(yPreds,yTrain)
    losses.append(loss.detach().numpy())

    
    loss.backward() #backprop
    optimizer.step() #updates parameters based on loss

    if(i%10==0): print(f"epoch={i}, loss={loss}")

/home/xyphoes/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([11610])) that is different to the input size (torch.Size([11610, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


epoch=0, loss=5.619421005249023
epoch=10, loss=5.443071365356445
epoch=20, loss=4.753010272979736
epoch=30, loss=3.980301856994629
epoch=40, loss=3.117631673812866


In [10]:
with torch.no_grad(): # won't calculate gradients
    yEvals = model.forwardProp(xTest)
    loss = lossFn(yEvals[:,0],yTest)\
    
print(loss)

tensor(2.0656)
